# 22 — Few-Shot Prompting

Static few-shot, dynamic example selection, and few-shot + structured output.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_chroma import Chroma
from pydantic import BaseModel, Field

## Example 1: Static Few-Shot

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
examples = [{"input": "happy", "output": "sad"}, {"input": "tall", "output": "short"}, {"input": "fast", "output": "slow"}, {"input": "bright", "output": "dim"}]
example_prompt = ChatPromptTemplate.from_messages([("human", "{input}"), ("ai", "{output}")])
few_shot = FewShotChatMessagePromptTemplate(example_prompt=example_prompt, examples=examples)
chain = ChatPromptTemplate.from_messages([("system", "You give the antonym of every word."), few_shot, ("human", "{input}")]) | llm | StrOutputParser()

for word in ["brave", "generous", "ancient"]:
    print(f"{word} → {chain.invoke({'input': word})}")

## Example 2: Dynamic Example Selection

In [ ]:
examples = [
    {"input": "What is 2+2?", "output": "The answer is 4."},
    {"input": "What is the capital of France?", "output": "The capital of France is Paris."},
    {"input": "How do I reverse a list in Python?", "output": "Use my_list[::-1] or my_list.reverse()."},
    {"input": "What is photosynthesis?", "output": "Plants convert sunlight into energy."},
    {"input": "How do I read a file in Python?", "output": "Use open('file.txt') with a context manager."},
    {"input": "What is gravity?", "output": "A fundamental force that attracts objects with mass."},
]

selector = SemanticSimilarityExampleSelector.from_examples(examples, OpenAIEmbeddings(model="text-embedding-3-small"), Chroma, k=2)
few_shot = FewShotChatMessagePromptTemplate(example_prompt=example_prompt, example_selector=selector)
chain = ChatPromptTemplate.from_messages([("system", "Answer concisely, following the example style."), few_shot, ("human", "{input}")]) | llm | StrOutputParser()

for q in ["How do I sort a dictionary in Python?", "What is the speed of light?", "What is 15 * 7?"]:
    selected = [ex["input"] for ex in selector.select_examples({"input": q})]
    print(f"Q: {q}\n  Examples: {selected}\n  A: {chain.invoke({'input': q})}\n")

## Example 3: Few-Shot + Structured Output

In [ ]:
class SentimentResult(BaseModel):
    sentiment: str = Field(description="positive, negative, or neutral")
    confidence: float
    key_phrase: str

examples = [
    {"input": "This product is absolutely amazing!", "output": "sentiment: positive, confidence: 0.95, key_phrase: 'absolutely amazing'"},
    {"input": "Terrible experience.", "output": "sentiment: negative, confidence: 0.90, key_phrase: 'Terrible experience'"},
    {"input": "It's okay, nothing special.", "output": "sentiment: neutral, confidence: 0.70, key_phrase: 'nothing special'"},
]
few_shot = FewShotChatMessagePromptTemplate(example_prompt=example_prompt, examples=examples)
chain = ChatPromptTemplate.from_messages([("system", "Analyse the sentiment."), few_shot, ("human", "{input}")]) | llm.with_structured_output(SentimentResult)

for text in ["I love how easy this is to use!", "The delivery was late and the item was damaged.", "The product works as described."]:
    r = chain.invoke({"input": text})
    print(f"{text}\n  → {r.sentiment} ({r.confidence:.0%}) — '{r.key_phrase}'\n")